In [ ]:
# RetailIQ Asia v2 — Databricks Data Load
# Load all retail benchmark data into Unity Catalog.
#
# Prerequisites:
# - Upload CSV files to /Volumes/<YOUR_CATALOG>/retailiq/retailiq/csv_files/
# - Upload .txt doc dirs to /Volumes/<YOUR_CATALOG>/retailiq/retailiq/doc_files/

In [ ]:
# Configuration
CATALOG = "<YOUR_CATALOG>"
SCHEMA = "retailiq"
CSV_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{SCHEMA}/csv_files"
DOC_PATH = f"/Volumes/{CATALOG}/{SCHEMA}/{SCHEMA}/doc_files"

spark.sql(f"USE CATALOG `{CATALOG}`")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SCHEMA}")
spark.sql(f"USE SCHEMA {SCHEMA}")

In [ ]:
# Load all 12 structured tables
tables = [
    ("dim_supplier", "dim_supplier.csv"),
    ("dim_store", "dim_store.csv"),
    ("dim_employee", "dim_employee.csv"),
    ("dim_product", "dim_product.csv"),
    ("fact_customer", "fact_customer.csv"),
    ("fact_order", "fact_order.csv"),
    ("fact_order_line", "fact_order_line.csv"),
    ("fact_return", "fact_return.csv"),
    ("fact_promotion", "fact_promotion.csv"),
    ("fact_inventory", "fact_inventory.csv"),
    ("fact_review", "fact_review.csv"),
    ("fact_loyalty_txn", "fact_loyalty_txn.csv"),
]

for table_name, csv_file in tables:
    print(f"Loading {table_name}...")
    df = (spark.read
          .option("header", "true")
          .option("inferSchema", "true")
          .option("nullValue", "")
          .csv(f"{CSV_PATH}/{csv_file}"))
    df.write.mode("overwrite").saveAsTable(f"`{CATALOG}`.{SCHEMA}.{table_name}")
    count = spark.table(f"`{CATALOG}`.{SCHEMA}.{table_name}").count()
    print(f"  → {table_name}: {count:,} rows")

print("\nAll structured tables loaded.")

In [ ]:
# Load documents from batched .txt files
# Upload docs_batched/ folders to:
#   /Volumes/<YOUR_CATALOG>/retailiq/retailiq/doc_files/products/
#   /Volumes/<YOUR_CATALOG>/retailiq/retailiq/doc_files/supplier_audits/
#   /Volumes/<YOUR_CATALOG>/retailiq/retailiq/doc_files/industry_research/

import os

DOC_VOL = f"/Volumes/{CATALOG}/{SCHEMA}/{SCHEMA}/doc_files"
SEPARATOR = "=" * 80 + "\n--- END OF DOCUMENT ---\n" + "=" * 80

collections = {
    "products": "product_description",
    "supplier_audits": "supplier_audit",
    "industry_research": "industry_research",
}

all_docs = []
for folder, doc_type in collections.items():
    folder_path = f"{DOC_VOL}/{folder}"
    txt_files = [f for f in os.listdir(folder_path) if f.endswith(".txt")]
    print(f"{folder}: {len(txt_files)} batch files")

    for txt_file in sorted(txt_files):
        with open(f"{folder_path}/{txt_file}", "r") as f:
            content = f.read()
        # Split by separator to get individual documents
        docs = content.split(SEPARATOR)
        for doc in docs:
            doc = doc.strip()
            if not doc:
                continue
            # First line is [FILE: original_name.txt]
            lines = doc.split("\n", 1)
            if lines[0].startswith("[FILE:"):
                fname = lines[0].replace("[FILE: ", "").replace("]", "").strip()
                text = lines[1].strip() if len(lines) > 1 else ""
            else:
                fname = "unknown.txt"
                text = doc
            doc_id = fname.replace(".txt", "")
            all_docs.append({
                "doc_id": doc_id,
                "doc_title": fname,
                "doc_type": doc_type,
                "doc_source": folder,
                "parsed_text": text,
            })
    print(f"  → {len([d for d in all_docs if d['doc_source'] == folder])} documents parsed")

print(f"\nTotal documents: {len(all_docs):,}")
doc_df = spark.createDataFrame(all_docs)
doc_df.write.mode("overwrite").saveAsTable(f"`{CATALOG}`.{SCHEMA}.doc_document")
count = spark.table(f"`{CATALOG}`.{SCHEMA}.doc_document").count()
print(f"doc_document table: {count:,} rows")

In [ ]:
# Verify all table row counts
all_tables = [t[0] for t in tables] + ["doc_document"]
for t in all_tables:
    count = spark.table(f"`{CATALOG}`.{SCHEMA}.{t}").count()
    print(f"{t}: {count:,}")